In [1]:
import torch
print(torch.__version__)  # должно быть что-то типа 2.2.1+cu121
print(torch.cuda.is_available())  # True
print(torch.cuda.get_device_name())  # NVIDIA GeForce RTX 3070

2.5.1+cu121
True
NVIDIA GeForce RTX 3070


In [2]:
import os
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json
from einops import rearrange

In [3]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
#
# class PatchEmbed(nn.Module):
#     def __init__(self, in_channels=3, embed_dim=768, tubelet_size=(2, 16, 16)):
#         super().__init__()
#         self.proj = nn.Conv3d(in_channels, embed_dim, kernel_size=tubelet_size, stride=tubelet_size)
#
#     def forward(self, x):
#         x = self.proj(x)          # [B, D, T', H', W']
#         x = x.flatten(2).transpose(1, 2)  # [B, N, D]
#         return x
#
# class ViViT(nn.Module):
#     def __init__(self,
#                  image_size=224,
#                  frames=16,
#                  patch_size=(2, 16, 16),
#                  in_channels=3,
#                  embed_dim=768,
#                  depth=8,
#                  num_heads=8,
#                  dropout=0.1):
#         super().__init__()
#
#         self.patch_embed = PatchEmbed(in_channels, embed_dim, patch_size)
#
#         num_patches = (frames // patch_size[0]) * (image_size // patch_size[1]) * (image_size // patch_size[2])
#         self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
#         self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
#         self.pos_dropout = nn.Dropout(p=dropout)
#
#         self.norm_pre = nn.LayerNorm(embed_dim)
#
#         encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads,
#                                                    dim_feedforward=embed_dim * 4,
#                                                    dropout=dropout, batch_first=True)
#         self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
#
#         self.head = nn.Sequential(
#             nn.LayerNorm(embed_dim),
#             nn.Linear(embed_dim, embed_dim // 2),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(embed_dim // 2, 1)
#         )
#
#         self._init_weights()
#
#     def _init_weights(self):
#         nn.init.normal_(self.cls_token, std=0.02)
#         nn.init.trunc_normal_(self.pos_embed, std=0.02)
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.xavier_uniform_(m.weight)
#                 if m.bias is not None:
#                     nn.init.constant_(m.bias, 0)
#
#     def forward(self, x):
#         x = x.permute(0, 4, 1, 2, 3)     # [B, C, T, H, W]
#         x = self.patch_embed(x)         # [B, N, D]
#
#         B, N, D = x.shape
#         cls_tokens = self.cls_token.expand(B, -1, -1)  # [B, 1, D]
#         x = torch.cat((cls_tokens, x), dim=1)          # [B, N+1, D]
#         x = x + self.pos_embed[:, :N+1, :]
#         x = self.pos_dropout(x)
#
#         x = self.norm_pre(x)
#         x = self.transformer(x)         # [B, N+1, D]
#         cls_output = x[:, 0]            # [B, D]
#         return self.head(cls_output).squeeze(-1)  # [B]

In [4]:
# class TinyViViT(nn.Module):
#     def __init__(self, in_channels=3, embed_dim=128, dropout=0.1):
#         super().__init__()
#
#         # Простейший Tubelet Embedding
#         self.embed = nn.Conv3d(in_channels, embed_dim, kernel_size=(2, 16, 16), stride=(2, 16, 16))
#
#         self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
#         self.pos_embed = nn.Parameter(torch.zeros(1, 27 + 1, embed_dim))  # 27 патчей при 224x224x16
#
#         encoder_layer = nn.TransformerEncoderLayer(
#             d_model=embed_dim, nhead=2, dim_feedforward=embed_dim*2,
#             dropout=dropout, batch_first=True
#         )
#         self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
#
#         self.norm = nn.LayerNorm(embed_dim)
#         self.head = nn.Linear(embed_dim, 1)
#
#     def forward(self, x):
#         x = x.permute(0, 4, 1, 2, 3)  # [B, C, T, H, W]
#         x = self.embed(x)             # [B, D, T', H', W']
#         x = x.flatten(2).transpose(1, 2)  # [B, N, D]
#
#         B, N, D = x.shape
#
#         if self.pos_embed.shape[1] != N + 1:
#             # 💡 динамически ресайзим pos_embed
#             self.pos_embed = nn.Parameter(
#                 F.interpolate(self.pos_embed[:, :1, :].transpose(1, 2), size=N + 1, mode='linear').transpose(1, 2)
#             )
#
#         cls = self.cls_token.expand(B, -1, -1)
#         x = torch.cat([cls, x], dim=1)                     # [B, N+1, D]
#         x = x + self.pos_embed[:, :x.size(1), :]           # [B, N+1, D]
#
#         x = self.transformer(x)
#         x = self.norm(x[:, 0])  # [B, D]
#         return self.head(x).squeeze(-1)
#


In [5]:
from sklearn.utils import shuffle

# Автоопределение последнего vN.npz
dataset_dir = "../dataset"
files = sorted([
    f for f in os.listdir(dataset_dir)
    if f.startswith("v") and f.endswith(".npz")
], key=lambda x: int(re.findall(r"v(\d+)", x)[0]))

dataset_path = os.path.join(dataset_dir, files[-1])
version = re.findall(r"v(\d+)", files[-1])[0]
print(f"[📦] Загружаем датасет: {dataset_path}")
data = np.load(dataset_path)
X = data["X"]  # [N, T, H, W, C]
y = data["y"]

X, y = shuffle(X, y, random_state=42)

[📦] Загружаем датасет: ../dataset\v4.npz


In [6]:
# X = X[:4]
# y = y[:4]

In [7]:
class VideoDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.astype(np.float32) / 255.0
        self.y = y.astype(np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.float32)


dataset = VideoDataset(X, y)
train_idx, val_idx = train_test_split(
    list(range(len(dataset))), test_size=0.2, stratify=y, random_state=42
)

b_s = 4
train_loader = DataLoader(torch.utils.data.Subset(dataset, train_idx), batch_size=b_s, shuffle=True)
val_loader = DataLoader(torch.utils.data.Subset(dataset, val_idx), batch_size=b_s)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
# import torch
# from torch.amp import GradScaler, autocast
#
# print(device)
# # model = ViViT(
# #     embed_dim=128,
# #     depth=2,
# #     num_heads=2,
# #     dropout=0.1
# # ).to(device)
# model = TinyViViT().to(device)
#
# criterion = nn.BCEWithLogitsLoss()
# optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
# scaler = GradScaler()
#
# EPOCHS = 13
#
# print("📚 Начинаем обучение модели ViViT...\n")
#
# for epoch in range(EPOCHS):
#     print(f"\n🔁 Эпоха {epoch + 1}/{EPOCHS}")
#     model.train()
#     total_loss, correct, total = 0.0, 0, 0
#
#     for i, (videos, labels) in enumerate(train_loader):
#         videos = videos.to(device)
#         labels = labels.float().to(device)
#
#         optimizer.zero_grad()
#
#         with autocast(device_type='cuda'):
#             outputs = model(videos)
#             loss = criterion(outputs, labels)
#
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#
#         total_loss += loss.item()
#         preds = (torch.sigmoid(outputs.detach()) > 0.5).int()
#         correct += (preds == labels.int()).sum().item()
#         total += labels.size(0)
#
#         # 🔄 Обновление строки батча
#         print(f"  🔹 Батч {i + 1}/{len(train_loader)}: Loss={loss.item():.4f}", end="\r", flush=True)
#
#     avg_loss = total_loss / len(train_loader)
#     acc = correct / total
#     print(" " * 60, end="\r")  # очистка строки
#     print(f"✅ Эпоха {epoch + 1} завершена — Train Acc: {acc:.2%}, Avg Loss: {avg_loss:.4f}")
#
#     # 🧪 Валидация на валидационном сете
#     model.eval()
#     val_correct, val_total = 0, 0
#     with torch.no_grad(), autocast(device_type='cuda'):
#         for videos, labels in val_loader:
#             videos = videos.to(device)
#             labels = labels.float().to(device)
#             outputs = model(videos)
#             preds = (torch.sigmoid(outputs) > 0.5).int()
#             val_correct += (preds == labels.int()).sum().item()
#             val_total += labels.size(0)
#
#     val_acc = val_correct / val_total
#     print(f"📊 Валид Acc: {val_acc:.2%}")
#
#     # 🧠 Диагностика: смотрим выход для первого видео
#     with torch.no_grad(), autocast(device_type='cuda'):
#         sample = dataset[0][0].unsqueeze(0).to(device)
#         output = model(sample)
#         print(f"🧪 Output logits: {output.item():.4f}, sigmoid: {torch.sigmoid(output).item():.4f}")

In [9]:
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from collections import defaultdict

# Модели, которые будем сравнивать
from torchvision.models.video import r3d_18, mc3_18, r2plus1d_18


class Conv3DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),
            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = self.net(x).view(x.size(0), -1)
        return self.fc(x).squeeze(-1)

results = defaultdict(list)

models_to_try = {
    "r3d_18": lambda: r3d_18(pretrained=False),
    "mc3_18": lambda: mc3_18(pretrained=False),
    "r2plus1d_18": lambda: r2plus1d_18(pretrained=False),
    "conv3d_custom": lambda: Conv3DCNN(),
}

In [21]:
import time
from sklearn.metrics import f1_score
import torch.nn.functional as F
import gc
from pathlib import Path

def train_and_eval(model_name, model_fn, epochs=5, results_dict=None):
    print(f"\n🧪 Обучаем {model_name}")
    model = model_fn()

    if hasattr(model, 'fc'):
        model.fc = nn.Linear(model.fc.in_features, 1)

    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    scaler = GradScaler()

    history = []

    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        all_preds, all_labels = [], []

        for videos, labels in train_loader:
            videos = videos.permute(0, 4, 1, 2, 3).to(device)  # [B, C, T, H, W]
            labels = labels.float().to(device)

            optimizer.zero_grad()
            with autocast(device_type="cuda"):
                outputs = model(videos).squeeze(-1)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            probs = torch.sigmoid(outputs.detach())
            preds = (probs > 0.5).int()

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.int().cpu().tolist())
            correct += (preds == labels.int()).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        f1 = f1_score(all_labels, all_preds, zero_division=0)
        mean_conf = torch.tensor(all_preds).float().mean().item()
        duration = time.time() - start_time

        print(f"  🔁 Эпоха {epoch+1}/{epochs} — Acc: {train_acc:.2%}, F1: {f1:.3f}, Conf: {mean_conf:.3f}, Loss: {total_loss:.4f}, Time: {duration:.1f}s")
        history.append((train_acc, f1, total_loss))

    # Валидация
    model.eval()
    val_correct, val_total = 0, 0
    val_preds, val_labels = [], []
    with torch.no_grad(), autocast(device_type="cuda"):
        for videos, labels in val_loader:
            videos = videos.permute(0, 4, 1, 2, 3).to(device)
            labels = labels.float().to(device)
            outputs = model(videos).squeeze(-1)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).int()
            val_preds.extend(preds.cpu().tolist())
            val_labels.extend(labels.int().cpu().tolist())
            val_correct += (preds == labels.int()).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    val_f1 = f1_score(val_labels, val_preds, zero_division=0)
    print(f"✅ {model_name} — Val Acc: {val_acc:.2%}, Val F1: {val_f1:.3f}")

    if results_dict is not None:
        results_dict[model_name] = {
            "train_acc": history[-1][0],
            "train_f1": history[-1][1],
            "val_acc": val_acc,
            "val_f1": val_f1,
            "loss": history[-1][2]
        }

    weights_dir = Path("weights") / model_name
    weights_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), weights_dir / "model.pth")

    del model
    torch.cuda.empty_cache()
    gc.collect()


In [22]:
from torchvision.models.video import r3d_18, mc3_18, r2plus1d_18

train_and_eval("r3d_18", lambda: r3d_18(weights=None), epochs = 5)


🧪 Обучаем r3d_18
  🔁 Эпоха 1/5 — Acc: 56.37%, F1: 0.557, Conf: 0.485, Loss: 71.5494, Time: 84.9s
  🔁 Эпоха 2/5 — Acc: 58.09%, F1: 0.571, Conf: 0.478, Loss: 69.6960, Time: 83.8s
  🔁 Эпоха 3/5 — Acc: 63.97%, F1: 0.633, Conf: 0.483, Loss: 64.5192, Time: 83.8s
  🔁 Эпоха 4/5 — Acc: 66.67%, F1: 0.665, Conf: 0.495, Loss: 60.5041, Time: 83.8s
  🔁 Эпоха 5/5 — Acc: 71.32%, F1: 0.713, Conf: 0.498, Loss: 56.8898, Time: 83.9s
✅ r3d_18 — Val Acc: 64.71%, Val F1: 0.640


In [23]:
train_and_eval("mc3_18", lambda: mc3_18(weights=None), epochs = 5)


🧪 Обучаем mc3_18
  🔁 Эпоха 1/5 — Acc: 56.13%, F1: 0.623, Conf: 0.664, Loss: 69.3238, Time: 109.0s
  🔁 Эпоха 2/5 — Acc: 62.75%, F1: 0.648, Conf: 0.559, Loss: 65.4372, Time: 108.6s
  🔁 Эпоха 3/5 — Acc: 65.44%, F1: 0.654, Conf: 0.498, Loss: 65.7042, Time: 108.5s
  🔁 Эпоха 4/5 — Acc: 71.57%, F1: 0.726, Conf: 0.539, Loss: 59.8802, Time: 108.6s
  🔁 Эпоха 5/5 — Acc: 66.42%, F1: 0.646, Conf: 0.449, Loss: 62.1933, Time: 108.6s
✅ mc3_18 — Val Acc: 67.65%, Val F1: 0.660


In [24]:
train_and_eval("r2plus1d_18", lambda: r2plus1d_18(weights=None), epochs = 5)


🧪 Обучаем r2plus1d_18
  🔁 Эпоха 1/5 — Acc: 53.43%, F1: 0.564, Conf: 0.569, Loss: 71.3685, Time: 255.6s
  🔁 Эпоха 2/5 — Acc: 58.33%, F1: 0.562, Conf: 0.451, Loss: 68.5423, Time: 255.2s
  🔁 Эпоха 3/5 — Acc: 62.75%, F1: 0.650, Conf: 0.564, Loss: 66.0264, Time: 255.3s
  🔁 Эпоха 4/5 — Acc: 66.67%, F1: 0.655, Conf: 0.466, Loss: 64.3079, Time: 255.4s
  🔁 Эпоха 5/5 — Acc: 67.16%, F1: 0.681, Conf: 0.529, Loss: 60.4042, Time: 255.2s
✅ r2plus1d_18 — Val Acc: 59.80%, Val F1: 0.705


In [25]:
train_and_eval("conv3d_custom", lambda: Conv3DCNN(), epochs = 5)


🧪 Обучаем conv3d_custom
  🔁 Эпоха 1/5 — Acc: 50.98%, F1: 0.642, Conf: 0.868, Loss: 70.7824, Time: 2.8s
  🔁 Эпоха 2/5 — Acc: 49.51%, F1: 0.112, Conf: 0.069, Loss: 70.7574, Time: 3.1s
  🔁 Эпоха 3/5 — Acc: 54.17%, F1: 0.370, Conf: 0.228, Loss: 70.5954, Time: 2.8s
  🔁 Эпоха 4/5 — Acc: 58.33%, F1: 0.472, Conf: 0.289, Loss: 70.5036, Time: 2.8s
  🔁 Эпоха 5/5 — Acc: 51.47%, F1: 0.108, Conf: 0.044, Loss: 70.3720, Time: 2.8s
✅ conv3d_custom — Val Acc: 62.75%, Val F1: 0.457
